In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import glob
import os


In [8]:
# Defining raw data files path 
data_folder = "../data/raw"

# Using glob to find all CSV files in that folder
all_files = glob.glob(os.path.join(data_folder, "*.csv"))

# Column validator
columns = ["ride_id", "rideable_type", "started_at", "ended_at", "start_station_name", "start_station_id",
            "end_station_name", "end_station_id", "start_lat", "start_lng", "end_lat", "end_lng", "member_casual"]

# Validating column names for files
validated_dfs = []
reference_dtypes = None
row_count = 0

for filename in all_files:
    df = pd.read_csv(filename)

    if list(df.columns) != columns:
        raise ValueError(f"Column mismatch in {filename}")
    
    # Set or Check Data Types
    if reference_dtypes is None:
        reference_dtypes = df.dtypes
        validated_dfs.append(df)
        row_count += len(df) 
    else:
        if df.dtypes.equals(reference_dtypes):
            validated_dfs.append(df)
            row_count += len(df)
        else:
            print(f"Mismatch in {filename}:")
            print(df.dtypes[df.dtypes != reference_dtypes])
            raise TypeError(f"Datatype mismatch in {filename}")

# Merging all validated files
combined_df = pd.concat(validated_dfs, ignore_index=True)

# Saving as merged_data.csv
output_filename = "../data/processed/merged_data.csv"

combined_df.to_csv(output_filename, index=False)

print(f"Successfully joined {len(validated_dfs)} files into '{output_filename}'!")

# Row count check
print(f"Indivdual file row_count: {row_count}")

Successfully joined 12 files into '../data/processed/merged_data.csv'!
Indivdual file row_count: 5552092


In [11]:
bike_data = pd.read_csv(output_filename)
print(f"Merged file row count: {len(bike_data)}")

Merged file row count: 5552092


In [13]:
df.head()

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual
0,D90034EFC7235A41,electric_bike,2025-12-22 17:20:27.872,2025-12-22 17:27:29.937,Broadway & Berwyn Ave,CHI00414,Broadway & Sunnyside Ave,CHI02089,41.978361,-87.659789,41.963419,-87.656069,member
1,B005230EE77A29D0,classic_bike,2025-12-20 17:15:31.749,2025-12-20 17:20:07.746,Sheridan Rd & Montrose Ave,CHI00290,Broadway & Sheridan Rd,CHI00476,41.961670,-87.654640,41.952833,-87.649993,member
2,FD33124F2231E215,classic_bike,2025-12-06 13:38:18.275,2025-12-06 13:52:54.682,Southport Ave & Wrightwood Ave,CHI00339,Larrabee St & Menomonee St,CHI00262,41.928773,-87.663913,41.914680,-87.643320,casual
3,7363165DB4BD6D14,electric_bike,2025-12-16 20:58:28.703,2025-12-16 21:04:35.476,Sheridan Rd & Montrose Ave,CHI00290,Clark St & Newport St,CHI00674,41.961670,-87.654640,41.944540,-87.654678,member
4,F09F745E43DF9556,electric_bike,2025-12-09 14:53:43.347,2025-12-09 14:58:30.944,Larrabee St & Kingsbury St 1,CHI00245,Rush St & Superior St,CHI00368,41.897764,-87.642884,41.895765,-87.625908,member


In [14]:
# Load merged data
df = pd.read_csv("../data/processed/merged_data.csv")

# Convert datetime columns
df["started_at"] = pd.to_datetime(df["started_at"])
df["ended_at"] = pd.to_datetime(df["ended_at"])

# Remove duplicate ride IDs
df = df.drop_duplicates(subset="ride_id")

# Remove rows with missing critical fields
df = df.dropna(subset=["started_at", "ended_at", "member_casual"])

# Create ride length (in minutes)
df["ride_length"] = (df["ended_at"] - df["started_at"]).dt.total_seconds() / 60

# Remove invalid ride durations
df = df[df["ride_length"] > 0]

# Create derived time features
df["day_of_week"] = df["started_at"].dt.day_name()
df["month"] = df["started_at"].dt.month_name()
df["hour"] = df["started_at"].dt.hour

# Save cleaned dataset
df.to_csv("../data/processed/cleaned_data.csv", index=False)

print("Data cleaning complete.")
print(f"Final row count: {len(df)}")

Data cleaning complete.
Final row count: 5552063
